[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/multilingual_text2cypher_evals.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

**致谢：** [Mohit Talniya](https://www.linkedin.com/in/mohit-talniya-5917b3242/)，Phoenix 大使

# 面向印度语言的多语言 Text2Cypher

评估流水线使用 Phoenix，为整个多语言评估过程提供全面的可观测性。从 Neo4j 数据集中的英文文本与 Cypher 配对出发，将问题翻译为目标语言，测试模型能否根据非英文输入生成正确的 Cypher。此方法支持：

- 端到端追踪“英文 → 翻译 → Cypher 生成”链路
- 使用回译和语义相似度评估翻译质量
- 对照英文标准答案比较跨语言 Cypher 准确率
- 捕获丰富元数据，支持深入分析失败原因
- 自动上传评估结果，支持团队协作

开始吧！



In [ ]:
!pip install dspy-ai pandas datasets sentence-transformers openinference-instrumentation-dspy openinference-instrumentation opentelemetry-api opentelemetry-sdk "litellm<1.82.7" openinference-instrumentation-litellm arize-phoenix-otel arize-phoenix-client

1. ***首先导入所需的全部库。***

In [ ]:
import os
from getpass import getpass

import dspy
from datasets import load_dataset
from openinference.instrumentation import suppress_tracing
from openinference.instrumentation.dspy import DSPyInstrumentor
from openinference.semconv.trace import SpanAttributes
from opentelemetry.trace import Status, StatusCode, format_span_id
from phoenix.client import Client
from phoenix.otel import register
from sentence_transformers import SentenceTransformer, util

# 全局配置
定义评估的主要参数，包括数据集大小、问题翻译的目标语言列表，以及待评估的语言模型。

In [ ]:
if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点： ")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥： ")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key


if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
os.environ["OPENAI_API_KEY"] = openai_api_key

PROJECT_NAME = "neo4j-multilingual-eval-v5"

# 评估运行配置
CONFIG = {"dataset_size": 10, "target_languages": ["Hindi", "Tamil", "Telugu"]}

# 定义所有待评估模型
# 注意：确保 API 基础 URL 对应本地模型服务器。
# 本 Notebook 使用 OpenAI，实际实现使用 gpt-oss-20b
models_to_evaluate = [
    # {
    #     "name": "openai/gpt-oss-20b",
    #     "instance": dspy.LM(model='openai/gpt-oss-20b', api_base="http://localhost:1234/v1"),
    #     "api_key_env": "OPENAI_API_KEY"
    # }
    {
        "name": "openai/gpt-4o-mini",
        "instance": dspy.LM(model="openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"]),
    }
]

# 定义初始化与验证函数
这些辅助函数用于初始化 Phoenix 追踪器以提供可观测性，并验证环境中是否具备所需的 API 密钥。

In [ ]:
def init_phoenix():
    """初始化 Phoenix 界面以获得可观测性。"""
    tracer_provider = register(
        project_name=PROJECT_NAME, endpoint=os.environ["PHOENIX_COLLECTOR_ENDPOINT"]
    )

    tracer = tracer_provider.get_tracer(__name__)
    DSPyInstrumentor().instrument(tracer_provider=tracer_provider)

    # LiteLLMInstrumentor().instrument(tracer_provider=tracer_provider)

    print("DSPy 插桩已启用")

    # 本地实例的 Phoenix register 要求端点包含 "/v1/traces"，而 tracer register 不需要
    base_url = os.environ["PHOENIX_COLLECTOR_ENDPOINT"].replace("/v1/traces", "")
    client = Client(base_url=base_url)
    return tracer, client

# 定义资源加载函数
此函数从 Hugging Face 加载 text2cypher 数据集和 sentence-transformer 模型，后者用于计算语义相似度以衡量翻译质量。

In [ ]:
def load_resources():
    """加载数据集与句子模型。"""
    print("正在加载 text2cypher 数据集")
    ds = load_dataset("neo4j/text2cypher-2024v1")
    train_samples = ds["train"].select(range(CONFIG["dataset_size"]))
    print(f"数据集已加载，样本数： {len(train_samples)} 个样本")

    print("正在加载用于语义评分的 sentence-transformer 模型")
    # 使用快速有效的 all-MiniLM-L6-v2 模型
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("句子模型加载成功")

    return train_samples, sentence_model

# 定义 DSPy 签名和流水线模块
使用 DSPy 签名系统定义 LLM 流水线的核心组件。

* Translate：将源文本翻译为目标语言。
* GenerateCypher：根据自然语言问题和数据库 schema 生成 Cypher 查询。
* EvaluateCypherEquivalence：使用 LLM 裁判判断两个 Cypher 查询在功能上是否等价。
* FullPipelineEvaluator：将翻译和 Cypher 生成串联起来的 dspy.Module。

In [ ]:
class Translate(dspy.Signature):
    """将文本翻译为目标语言。"""

    source_text = dspy.InputField(desc="源语言问题")
    target_language = dspy.InputField(desc="语言名称")
    translated_text = dspy.OutputField(desc="目标语言问题。")


class GenerateCypher(dspy.Signature):
    """根据问题与模式生成 Cypher 查询。"""

    question = dspy.InputField(desc="自然语言问题")
    neo4j_schema = dspy.InputField(desc="Neo4J schema")
    cypher_query = dspy.OutputField(desc="生成的 Cypher 查询")


class EvaluateCypherEquivalence(dspy.Signature):
    """通过类别评估判断两个 Cypher 查询是否功能等价。"""

    ground_truth_query = dspy.InputField(desc="正确的参考 Cypher 查询")
    generated_query = dspy.InputField(desc="待评估的生成 Cypher 查询")
    natural_lang_query = dspy.InputField(desc="英文自然语言查询")
    neo4j_schema = dspy.InputField(desc="供参考的 Neo4j 数据库模式")
    equivalence_category = dspy.OutputField(
        desc="以下之一：EQUIVALENT、PARTIALLY_CORRECT、INCORRECT、SYNTAX_ERROR"
    )
    reasoning = dspy.OutputField(desc="分类的简要说明")


class FullPipelineEvaluator(dspy.Module):
    """执行翻译与 Cypher 生成步骤的 DSPy 模块。"""

    def __init__(self):
        super().__init__()
        self.translator = dspy.Predict(Translate)
        self.cypher_generator = dspy.Predict(GenerateCypher)

    def forward(self, question, schema, target_language):
        print(f"正在为以下问题运行流水线： {question[:50]}...")

        # 1. 获取翻译
        t = self.translator(source_text=question, target_language=target_language)
        print(f"翻译完成： {t.translated_text[:50]}...")

        # 2. 根据翻译后的问题生成 Cypher
        c = self.cypher_generator(question=t.translated_text, neo4j_schema=schema)
        print(f"Cypher 生成完成： {c.cypher_query[:50]}...")

        return dspy.Prediction(
            translated_question=t.translated_text, generated_query=c.cypher_query
        )

# 定义评估函数
这些函数负责为流水线输出评分。

* score_translation_quality：通过回译和语义相似度评估翻译质量。
* compare_query_results_with_llm：使用 EvaluateCypherEquivalence 裁判评估生成的 Cypher 查询是否正确。
* safe_set_span_attributes：安全地为 OpenTelemetry span 添加追踪属性的工具函数。

In [ ]:
def score_translation_quality(original_text, translated_text, back_translator, sentence_model):
    """执行回译并返回语义相似度分数。"""
    print("正在评估翻译质量")
    with suppress_tracing():
        try:
            back_translation_result = back_translator(
                source_text=translated_text, target_language="English"
            )
            back_translated_text = back_translation_result.translated_text

            embedding_1 = sentence_model.encode(original_text, convert_to_tensor=True)
            embedding_2 = sentence_model.encode(back_translated_text, convert_to_tensor=True)

            similarity_score = util.pytorch_cos_sim(embedding_1, embedding_2).item()
            print(f"翻译质量分数： {similarity_score:.3f}")

            return {"score": similarity_score, "back_translated_text": back_translated_text}
        except Exception as e:
            print(f"翻译质量评估出错： {e}")
            return {"score": 0.0, "back_translated_text": "", "error": str(e)}


def compare_query_results_with_llm(
    ground_truth_query, generated_query, schema, cypher_judge, natural_language_question
):
    """通过分类 LLM 裁判比较查询。"""
    print("正在通过分类 LLM 裁判比较查询")
    try:
        result = cypher_judge(
            ground_truth_query=ground_truth_query,
            generated_query=generated_query,
            neo4j_schema=schema,
            natural_lang_query=natural_language_question,
        )

        category = result.equivalence_category.upper()

        if category == "EQUIVALENT":
            correct, score = True, 1.0
        elif category == "PARTIALLY_CORRECT":
            correct, score = None, 0.5  # 部分得分
        else:  # INCORRECT、SYNTAX_ERROR 或其他情况
            correct, score = False, 0.0

        return {
            "correct": correct,
            "category": category,
            "score": score,
            "reason": result.reasoning,
        }

    except Exception as e:
        print(f"LLM 裁判评估出错： {e}")
        return {
            "correct": False,
            "category": "ERROR",
            "score": 0.0,
            "reason": f"裁判评估错误： {e}",
        }


def safe_set_span_attributes(span, attributes):
    """安全设置跨度属性，避免触发警告。"""
    if span and span.is_recording():
        try:
            span.set_attributes(attributes)
        except Exception as e:
            print(f"无法设置跨度属性： {e}")

# 定义样本处理核心函数
这是主要的编排函数。它针对单个样本运行完整流水线、评估结果，并将全部信息记录到 Phoenix 中可见的自定义 OpenTelemetry span。

In [ ]:
def process_and_evaluate_sample(
    tracer,
    sample,
    evaluator,
    back_translator,
    sentence_model,
    target_language,
    cypher_judge,
    sample_id,
):
    """处理单个样本并编排全部步骤的主函数。"""
    print(f"正在处理样本 {sample_id}，语言： {target_language}")

    with tracer.start_as_current_span(
        f"multilingual_evaluation_{target_language}",
        openinference_span_kind="chain",
        attributes={"evaluation.sample_id": str(sample_id)},
    ) as eval_span:
        try:
            # 运行完整翻译与 Cypher 生成流水线
            print(f"原始问题（英文）： {sample['question']}")
            pipeline_result = evaluator(
                question=sample["question"],
                schema=sample["schema"],
                target_language=target_language,
            )
            print(f"翻译后的问题（{target_language}): {pipeline_result.translated_question}")

            # 评估翻译质量
            translation_score_result = score_translation_quality(
                sample["question"],
                pipeline_result.translated_question,
                back_translator,
                sentence_model,
            )

            # 评估 Cypher 查询正确性
            cypher_assessment = compare_query_results_with_llm(
                sample["cypher"],
                pipeline_result.generated_query,
                sample["schema"],
                cypher_judge,
                sample["question"],
            )

            # 将所有评估结果设置为跨度属性
            evaluation_attrs = {
                "evaluation.translation_score": float(translation_score_result.get("score", 0.0)),
                "evaluation.cypher_correct": cypher_assessment.get("correct"),
                "evaluation.cypher_category": cypher_assessment.get("category", ""),
                "evaluation.cypher_score": cypher_assessment.get("score", 0.0),
                "evaluation.cypher_reason": cypher_assessment.get("reason", ""),
                "evaluation.ground_truth_query": sample["cypher"],
                "evaluation.generated_query": pipeline_result.generated_query,
                "evaluation.original_text": sample["question"],
                "evaluation.translated_text": pipeline_result.translated_question,
                SpanAttributes.INPUT_VALUE: pipeline_result.translated_question,
                SpanAttributes.OUTPUT_VALUE: pipeline_result.generated_query,
            }
            safe_set_span_attributes(eval_span, evaluation_attrs)
            eval_span.set_status(Status(StatusCode.OK))
            span_id = format_span_id(eval_span.get_span_context().span_id)

            print(f"样本 {sample_id} 评估成功完成")
            return pipeline_result, translation_score_result, cypher_assessment, span_id

        except Exception as e:
            print(f"处理样本出错： {sample_id}: {e}", exc_info=True)
            safe_set_span_attributes(eval_span, {"error": str(e)})
            eval_span.set_status(Status(StatusCode.ERROR, description=str(e)))
            span_id = format_span_id(eval_span.get_span_context().span_id)
            return (
                None,
                {"score": 0.0, "error": str(e)},
                {"correct": False, "reason": f"流水线错误： {e}"},
                span_id,
            )

# 初始化并运行完整评估
这是主要执行代码块。初始化 Phoenix、验证密钥并加载数据，然后遍历各模型、各目标语言及各数据样本，调用上方的处理函数并收集结果。

In [ ]:
# 1. 初始化 Phoenix 并验证配置
tracer, client = init_phoenix()

# 2. 加载数据集与句子模型
train_samples, sentence_model = load_resources()

all_results = []

# 3. 主评估循环
for model_config in models_to_evaluate:
    model_name = model_config["name"]
    llm_instance = model_config["instance"]

    print(f"--- 正在评估模型： {model_name} ---")
    dspy.settings.configure(lm=llm_instance)

    # 为此模型初始化 DSPy 组件
    evaluator = FullPipelineEvaluator()
    back_translator = dspy.Predict(Translate)
    cypher_judge = dspy.Predict(EvaluateCypherEquivalence)

    for target_language in CONFIG["target_languages"]:
        print(f"--- 正在测试语言： {target_language} ---")

        for i, sample in enumerate(train_samples):
            print(f"正在处理样本 {i + 1}/{len(train_samples)}，对应 {target_language}")

            pipeline_result, translation_score, cypher_assessment, span_id = (
                process_and_evaluate_sample(
                    tracer=tracer,
                    sample=sample,
                    evaluator=evaluator,
                    back_translator=back_translator,
                    sentence_model=sentence_model,
                    target_language=target_language,
                    cypher_judge=cypher_judge,
                    sample_id=i,
                )
            )

            # 在 Phoenix 中为跨度添加标注
            try:
                client.spans.add_span_annotation(
                    annotation_name="translation_score",
                    annotator_kind="LLM",
                    span_id=span_id,
                    label="translation",
                    score=float(translation_score.get("score", 0.0)),
                )
                client.spans.add_span_annotation(
                    annotation_name="cypher_score",
                    annotator_kind="LLM",
                    span_id=span_id,
                    label="cypher",
                    score=cypher_assessment.get("score", 0.0),
                    metadata={"category": cypher_assessment.get("category", "")},
                )
            except Exception as e:
                print(f"推送以下跨度的标注失败： {span_id}: {e}")

            # 保存结果以生成最终汇总报告
            result = {
                "model_name": model_name,
                "sample_id": i,
                "question": sample["question"],
                "target_language": target_language,
                "translation_score": translation_score.get("score", 0.0),
                "is_correct": cypher_assessment.get("correct"),
                "cypher_category": cypher_assessment.get("category"),
                "cypher_score": cypher_assessment.get("score", 0.0),
                "reason": cypher_assessment.get("reason"),
                "has_error": "error" in translation_score or "error" in cypher_assessment,
            }
            if pipeline_result:
                result.update(
                    {
                        "translated_question": pipeline_result.translated_question,
                        "generated_query": pipeline_result.generated_query,
                    }
                )
            all_results.append(result)

print("--- 完整评估运行结束。 ---")

# ![Phoenix Neo4j 追踪](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/neo4j_evals/phoenix_neo4j.png)



# ![Phoenix Neo4j 标注预览](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/neo4j_evals/phoenix_neo4j2.png)
